# Análisis y visualización de resultados DEI

Este notebook genera las visualizaciones y responde a las preguntas de investigación a partir de los resultados almacenados en la base de datos.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mtick
import numpy as np

from modules.config import engine
from modules.patterns import dei_patterns
from modules.visualization import (
    grafico_frecuencia_dei,
    grafico_dei_por_metodo,
    grafico_dei_por_continente,
    grafico_evolucion_temporal,
)

mpl.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 13,
})

## 1. Carga de datos

In [ ]:
df = pd.read_sql("SELECT * FROM cases_dei_metrics", engine)
df_cases = pd.read_sql("SELECT id, country, continent FROM cases", engine)
df_merged = df.merge(df_cases, on='id')

cols_cat = [cat for cat in dei_patterns]


## 2. Presencia general de categorías DEI

In [ ]:
df = pd.read_sql("""
    SELECT 
        COUNT(DISTINCT c.id) as total_casos,
        COUNT(DISTINCT g.case_id) as casos_con_dei,
        COUNT(DISTINCT c.id) - COUNT(DISTINCT g.case_id) as casos_sin_dei
    FROM cases c
    LEFT JOIN cases_llm_groups g ON g.case_id = c.id 
        AND g.role != 'false_positive'
""", engine)

total = df['total_casos'].values[0]
con_dei = df['casos_con_dei'].values[0]
sin_dei = df['casos_sin_dei'].values[0]

print(f"Total casos: {total}")
print(f"Con DEI: {con_dei} ({con_dei/total*100:.1f}%)")
print(f"Sin DEI: {sin_dei} ({sin_dei/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie([con_dei, sin_dei], 
       labels=[f'Con categorías DEI\n({con_dei/total*100:.1f}%)', 
               f'Sin categorías DEI\n({sin_dei/total*100:.1f}%)'],
       colors=['#1D9E75', '#d0d0d0'],
       autopct='%1.1f%%',
       startangle=90,
       textprops={'fontsize': 12})
ax.set_title('Proporción de casos de Participedia\ncon categorías DEI detectadas', fontsize=13)

plt.tight_layout()
plt.savefig('proporcion_dei.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: proporcion_dei.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Leer el CSV — ajusta el nombre si es diferente
df = pd.read_sql("""SELECT 
    category,
    COUNT(DISTINCT case_id) as casos,
    ROUND(100.0 * COUNT(DISTINCT case_id) / (SELECT COUNT(*) FROM cases), 1) as pct_total
FROM cases_llm_groups
WHERE role != 'false_positive'
GROUP BY category
ORDER BY casos DESC""", engine)

df = df.sort_values('casos', ascending=True)

df['category'] = df['category'].str.replace('cat_', '')

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(df['category'], df['casos'], color='#1D9E75', edgecolor='white', linewidth=0.5)

for bar, (_, row) in zip(bars, df.iterrows()):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height() / 2,
            f"{int(row['casos'])} ({row['pct_total']:.1f}%)",
            va='center', fontsize=9, color='#444441')

ax.set_xlabel('Nº de casos', fontsize=11)
ax.set_xlim(0, df['casos'].max() * 1.2)

plt.tight_layout()
plt.savefig('frecuencia_categorias_dei.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: frecuencia_categorias_dei.png")

In [ ]:
import matplotlib.ticker as mtick

df = pd.read_sql("""SELECT 
    n_categorias,
    COUNT(*) as total_casos,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as porcentaje
FROM (
    SELECT case_id, COUNT(DISTINCT category) as n_categorias
    FROM cases_llm_groups
    WHERE role != 'false_positive'
    GROUP BY case_id
) t
GROUP BY n_categorias
ORDER BY n_categorias;
""", engine

)
fig, ax = plt.subplots(figsize=(10, 6))

colores = ['#888780' if x == 0 else '#1D9E75' if x == 1 else '#534AB7' if x <= 3 else '#D85A30'
           for x in df['n_categorias']]

bars = ax.bar(df['n_categorias'], df['total_casos'], color=colores, edgecolor='white', linewidth=0.5)

for bar, (_, row) in zip(bars, df.iterrows()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
            f"{int(row['total_casos'])}\n({row['porcentaje']:.1f}%)",
            ha='center', fontsize=9, color='#444441')

ax.set_xlabel('Nº de categorías DEI mencionadas', fontsize=11)
ax.set_ylabel('Nº de casos', fontsize=11)
ax.set_xticks(df['n_categorias'])

plt.tight_layout()
plt.savefig('profundidad_dei.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: profundidad_dei.png")

## 3. Grupos sociales y roles

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from modules.config import engine

df = pd.read_sql("""
    SELECT 
        role,
        COUNT(*) as total,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as porcentaje
    FROM cases_llm_groups
    WHERE role NOT IN ('false_positive', 'historical_context')
    GROUP BY role
    ORDER BY total DESC
""", engine)

traducciones = {
    'participant': 'Participante',
    'target': 'Objetivo',
    'design_criterion': 'Criterio de diseño'
}
df['role_es'] = df['role'].map(traducciones)

fig, ax = plt.subplots(figsize=(8, 8))
colores = ['#1D9E75', '#D85A30', '#534AB7']
wedges, texts, autotexts = ax.pie(
    df['total'],
    labels=df['role_es'],
    colors=colores,
    autopct='%1.1f%%',
    startangle=90,
    textprops={'fontsize': 12}
)


plt.tight_layout()
plt.savefig('distribucion_roles.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: distribucion_roles.png")

In [ ]:
df = pd.read_sql("""
    WITH mapped_cases AS (
        SELECT case_id, role,
            CASE 
                WHEN group_name IN (
                    '11-15-year-olds', '7 -11 year olds', 'adolescents', 'at risk teenagers', 
                    'boys', 'children', 'children with disability', 'elementary and middle schoolers', 
                    'girls', 'handicapped children', 'high school students', 'kids', 
                    'looked after children', 'minors', 'teenagers', 'very young', 
                    'vulnerable children', 'young girls',
                    '15-29', 'under 20', 'younger than 25', 'youth citizens below 26 years old', 
                    'transition age youth', 'teen parents', 'teenage mothers', 'vulnerable youth', 
                    'people under 30 years old', 'people over 16', 'residents aged 16 or more', 
                    'ages 15 to 80', 'citizens over 16 or over 70'
                ) THEN '0-17'
                WHEN group_name IN (
                    'heiltsuk youths', 'immigrant youth', 'indigenous young people', 'neet youth', 
                    'recent graduates', 'unemployed youth', 'young adults', 'young individuals', 
                    'young men', 'young moroccans', 'young people', 'young people with a learning disability', 
                    'young people with a physical or learning disability', 'young professionals', 
                    'young refugees', 'young voters', 'young women', 'young workers', 
                    'youth', 'youth experiencing homelessness', 'youth facing mental health challenges', 
                    'youth who are uninsured', 'youth with different religious affiliations', 'youth with disabilities',
                    '25-35 year olds', '25-40', '30-49', 'people aged 25 to 44', 
                    'citizens aged 18 to 70', 'people over the age of 18'
                ) THEN '18-30'
                WHEN group_name IN (
                    '40-60', 'adults', 'middle aged', 'millennials', 
                    'people in their 30''s', 'people in their 40''s', 'people in their 50''s',
                    '50-64', 'over 50', 'participants aged over 50', 'people aged 40-64'
                ) THEN '31-60'
                WHEN group_name IN (
                    'elderly', 'homebound seniors', 'old', 'older adults', 
                    'older generation', 'older members', 'older people', 'older women', 
                    'pensioners', 'people in their 60''s', 'people in their eighties', 'retirees', 
                    'senior citizens', 'seniors'
                ) THEN '60+'
            END as franja_edad
        FROM cases_llm_groups
        WHERE category = 'cat_group_age'
        AND role IN ('participant', 'target', 'design_criterion')
        AND group_name != ''
    )
    SELECT franja_edad, role, COUNT(DISTINCT case_id) as total_casos
    FROM mapped_cases
    WHERE franja_edad IS NOT NULL
    GROUP BY franja_edad, role
    ORDER BY franja_edad, role
""", engine)

df_pivot = df.pivot(index='franja_edad', columns='role', values='total_casos').fillna(0)

orden = ['0-17', '18-30', '31-60', '60+']
df_pivot = df_pivot.reindex([o for o in orden if o in df_pivot.index])

import seaborn as sns
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(df_pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_xlabel('Rol', fontsize=11)
ax.set_ylabel('Franja de edad', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('heatmap_edad_rol.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_sql("""WITH mapped_cases AS (
    -- 1. REGISTROS QUE PERTENECEN O CRUZAN A LA FRANJA '0-17'
    SELECT case_id, '0-17' as franja_edad 
    FROM cases_llm_groups 
    WHERE category = 'cat_group_age' AND role != 'false_positive' AND group_name != ''
    AND group_name IN (
        '11-15-year-olds', '7 -11 year olds', 'adolescents', 'at risk teenagers', 
        'boys', 'children', 'children with disability', 'elementary and middle schoolers', 
        'girls', 'handicapped children', 'high school students', 'kids', 
        'looked after children', 'minors', 'teenagers', 'very young', 
        'vulnerable children', 'young girls',
        -- Grupos que cruzan con '18-30' o rangos macro:
        '15-29', 'under 20', 'younger than 25', 'youth citizens below 26 years old', 
        'transition age youth', 'teen parents', 'teenage mothers', 'vulnerable youth', 
        'people under 30 years old', 'people over 16', 'residents aged 16 or more', 
        'ages 15 to 80', 'citizens over 16 or over 70'
    )

    UNION ALL

    -- 2. REGISTROS QUE PERTENECEN O CRUZAN A LA FRANJA '18-30'
    SELECT case_id, '18-30' as franja_edad 
    FROM cases_llm_groups 
    WHERE category = 'cat_group_age' AND role != 'false_positive' AND group_name != ''
    AND group_name IN (
        'heiltsuk youths', 'immigrant youth', 'indigenous young people', 'neet youth', 
        'recent graduates', 'unemployed youth', 'young adults', 'young individuals', 
        'young men', 'young moroccans', 'young people', 'young people with a learning disability', 
        'young people with a physical or learning disability', 'young professionals', 
        'young refugees', 'young voters', 'young women', 'young workers', 
        'youth', 'youth experiencing homelessness', 'youth facing mental health challenges', 
        'youth who are uninsured', 'youth with different religious affiliations', 'youth with disabilities',
        -- Grupos que cruzan desde '0-17':
        '15-29', 'under 20', 'younger than 25', 'youth citizens below 26 years old', 
        'transition age youth', 'teen parents', 'teenage mothers', 'vulnerable youth', 
        'people under 30 years old',
        -- Grupos que cruzan hacia '31-60' o rangos macro:
        '25-35 year olds', '25-40', '30-49', 'people aged 25 to 44', 'people over 16', 
        'residents aged 16 or more', 'ages 15 to 80', 'citizens aged 18 to 70', 
        'citizens over 16 or over 70', 'people over the age of 18'
    )

    UNION ALL

    -- 3. REGISTROS QUE PERTENECEN O CRUZAN A LA FRANJA '31-60'
    SELECT case_id, '31-60' as franja_edad 
    FROM cases_llm_groups 
    WHERE category = 'cat_group_age' AND role != 'false_positive' AND group_name != ''
    AND group_name IN (
        '40-60', 'adults', 'middle aged', 'millennials', 
        'people in their 30’s', 'people in their 40’s', 'people in their 50’s',
        -- Grupos que cruzan desde '18-30':
        '25-35 year olds', '25-40', '30-49', 'people aged 25 to 44',
        -- Grupos que cruzan hacia '60+' o rangos macro:
        '50-64', 'over 50', 'participants aged over 50', 'people aged 40-64', 
        'people over 16', 'residents aged 16 or more', 'ages 15 to 80', 
        'citizens aged 18 to 70', 'citizens over 16 or over 70', 'people over the the age of 18'
    )

    UNION ALL

    -- 4. REGISTROS QUE PERTENECEN O CRUZAN A LA FRANJA '60+'
    SELECT case_id, '60+' as franja_edad 
    FROM cases_llm_groups 
    WHERE category = 'cat_group_age' AND role != 'false_positive' AND group_name != ''
    AND group_name IN (
        'elderly', 'homebound seniors', 'old', 'older adults', 
        'older generation', 'older members', 'older people', 'older women', 
        'pensioners', 'people in their 60’s', 'people in their eighties', 'retirees', 
        'senior citizens', 'seniors',
        -- Grupos que cruzan desde '31-60' o rangos macro:
        '50-64', 'over 50', 'participants aged over 50', 'people aged 40-64', 
        'people over 16', 'residents aged 16 or more', 'ages 15 to 80', 
        'citizens aged 18 to 70', 'citizens over 16 or over 70', 'people over the age of 18'
    )

    UNION ALL

    -- 5. GRUPOS EXCLUIDOS POR NO SER CATEGORÍAS DE EDAD (Van a 'otros')
    SELECT case_id, 'otros' as franja_edad 
    FROM cases_llm_groups 
    WHERE category = 'cat_group_age' AND role != 'false_positive' AND group_name != ''
    AND group_name IN (
        'asian american student union', 'care leavers', 'carers with disability', 
        'community members', 'disadvantaged students', 'female students', 
        'fulltime mothers with children of school age', 'homeschool parents', 
        'intergenerational', 'local residents', 'male students', 'parents', 
        'parents of school-age children', 'participants', 'students'
    )
)
SELECT 
    franja_edad,
    COUNT(DISTINCT case_id) as total_casos
FROM mapped_cases
GROUP BY franja_edad
ORDER BY total_casos DESC""", engine)

orden = ['0-17', '18-30', '31-60', '60+', 'Multigeneracional']
df = df[df['franja_edad'].isin(orden)].copy()
df['franja_edad'] = pd.Categorical(df['franja_edad'], categories=orden, ordered=True)
df = df.sort_values('franja_edad')

fig, ax = plt.subplots(figsize=(10, 2))

valores = df['total_casos'].values.reshape(1, -1)
im = ax.imshow(valores, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(df)))
ax.set_xticklabels(df['franja_edad'], fontsize=11)
ax.set_yticks([])

for j, val in enumerate(df['total_casos']):
    ax.text(j, 0, str(int(val)), ha='center', va='center', 
            fontsize=11, fontweight='bold', color='black')

plt.colorbar(im, ax=ax, label='Nº de casos', orientation='vertical')
ax.set_title('Atención a grupos por franja de edad en Participedia', fontsize=13, pad=15)

plt.tight_layout()
plt.savefig('heatmap_franjas_edad.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Variación geográfica

In [ ]:
df_diversidad_continente = pd.read_sql("""
    SELECT 
        c.continent,
        COUNT(DISTINCT c.id) as total_casos,
        COUNT(DISTINCT g.case_id) as casos_con_dei,
        ROUND(100.0 * COUNT(DISTINCT g.case_id) / COUNT(DISTINCT c.id), 1) as pct_con_dei
    FROM cases c
    LEFT JOIN cases_llm_groups g ON g.case_id = c.id 
        AND g.role != 'false_positive'
    WHERE c.continent IS NOT NULL
    GROUP BY c.continent
    ORDER BY pct_con_dei DESC
""", engine)

print(df_diversidad_continente.to_string(index=False))

In [ ]:
df_continent = pd.read_sql("""
    SELECT 
        c.continent,
        COUNT(DISTINCT c.id) as total_casos,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_gender = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_gender' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_gender,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_race = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_race' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_race,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_class = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_class' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_class,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_group_age = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_group_age' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_group_age,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_disability = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_disability' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_disability,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_migration = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_migration' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_migration,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_territory = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_territory' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_territory,
        ROUND(100.0 * COUNT(DISTINCT CASE WHEN m.cat_religion = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id 
                       AND g.category = 'cat_religion' 
                       AND g.role != 'false_positive')
            THEN c.id END) / COUNT(DISTINCT c.id), 1) as pct_religion
    FROM cases c
    JOIN cases_dei_metrics m ON c.id = m.id
    WHERE c.continent IS NOT NULL
    GROUP BY c.continent
    ORDER BY total_casos DESC
""", engine)

grafico_dei_por_continente(df_continent)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from modules.config import engine

df = pd.read_sql("""
    SELECT 
        c.continent,
        m.title as metodo,
        COUNT(DISTINCT c.id) as total_casos
    FROM cases c
    JOIN cases_methods cm ON c.id = cm.case_id
    JOIN methods m ON m.id = cm.method_id
    WHERE c.continent IS NOT NULL
    GROUP BY c.continent, m.title
    ORDER BY c.continent, total_casos DESC
""", engine)

df_top = df.groupby('continent').apply(
    lambda x: x.nlargest(3, 'total_casos')
).reset_index(drop=True)

continentes = df_top['continent'].unique()
fig, axes = plt.subplots(len(continentes), 1, figsize=(10, len(continentes) * 3))

for ax, continente in zip(axes, continentes):
    df_cont = df_top[df_top['continent'] == continente]
    ax.barh(df_cont['metodo'], df_cont['total_casos'], color='#1D9E75')
    ax.set_title(continente, fontsize=11, fontweight='bold')
    ax.set_xlabel('Nº de casos')
    for i, v in enumerate(df_cont['total_casos']):
        ax.text(v + 0.5, i, str(v), va='center', fontsize=9)

plt.suptitle('Top 3 métodos participativos por continente', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('metodos_por_continente.png', dpi=150, bbox_inches='tight')
plt.show()
print("Guardado: metodos_por_continente.png")

## 5. Evolución temporal

In [ ]:
df_temporal = pd.read_sql("""
    SELECT 
        EXTRACT(YEAR FROM c.start_date) as año,
        COUNT(DISTINCT c.id) as total_casos,
        COUNT(DISTINCT CASE WHEN m.cat_gender = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id AND g.category = 'cat_gender' 
                       AND g.role != 'false_positive')
            THEN c.id END) as casos_gender,
        COUNT(DISTINCT CASE WHEN m.cat_race = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id AND g.category = 'cat_race' 
                       AND g.role != 'false_positive')
            THEN c.id END) as casos_race,
        COUNT(DISTINCT CASE WHEN m.cat_class = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id AND g.category = 'cat_class' 
                       AND g.role != 'false_positive')
            THEN c.id END) as casos_class,
        COUNT(DISTINCT CASE WHEN m.cat_group_age = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id AND g.category = 'cat_group_age' 
                       AND g.role != 'false_positive')
            THEN c.id END) as casos_group_age,
        COUNT(DISTINCT CASE WHEN m.cat_disability = 1 
            AND EXISTS (SELECT 1 FROM cases_llm_groups g 
                       WHERE g.case_id = c.id AND g.category = 'cat_disability' 
                       AND g.role != 'false_positive')
            THEN c.id END) as casos_disability
    FROM cases c
    JOIN cases_dei_metrics m ON c.id = m.id
    WHERE c.start_date IS NOT NULL and c.start_date >= '1999-12-31'
    AND EXTRACT(YEAR FROM c.start_date) BETWEEN 1990 AND 2024
    GROUP BY año
    ORDER BY año
""", engine)

# Calcular porcentajes
for cat in ['gender', 'race', 'class', 'group_age', 'disability']:
    df_temporal[f'pct_{cat}'] = (df_temporal[f'casos_{cat}'] / df_temporal['total_casos'] * 100).round(1)

print(df_temporal[['año', 'total_casos', 'pct_gender', 'pct_race', 'pct_class', 'pct_group_age', 'pct_disability']])
grafico_evolucion_temporal(df_temporal)

## 6. DEI por tipo de método participativo

In [ ]:
df = pd.read_sql("""
    SELECT m.title as metodo,
           COUNT(DISTINCT c.id) as total_casos,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN EXISTS (SELECT 1 FROM cases_llm_groups g WHERE g.case_id = c.id AND g.category = 'cat_gender' AND g.role IN ('participant', 'design_criterion')) THEN c.id END) / COUNT(DISTINCT c.id), 1) as gender,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN EXISTS (SELECT 1 FROM cases_llm_groups g WHERE g.case_id = c.id AND g.category = 'cat_race' AND g.role IN ('participant', 'design_criterion')) THEN c.id END) / COUNT(DISTINCT c.id), 1) as race,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN EXISTS (SELECT 1 FROM cases_llm_groups g WHERE g.case_id = c.id AND g.category = 'cat_class' AND g.role IN ('participant', 'design_criterion')) THEN c.id END) / COUNT(DISTINCT c.id), 1) as class,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN EXISTS (SELECT 1 FROM cases_llm_groups g WHERE g.case_id = c.id AND g.category = 'cat_group_age' AND g.role IN ('participant', 'design_criterion')) THEN c.id END) / COUNT(DISTINCT c.id), 1) as group_age,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN EXISTS (SELECT 1 FROM cases_llm_groups g WHERE g.case_id = c.id AND g.category = 'cat_disability' AND g.role IN ('participant', 'design_criterion')) THEN c.id END) / COUNT(DISTINCT c.id), 1) as disability,
           ROUND(100.0 * COUNT(DISTINCT CASE WHEN EXISTS (SELECT 1 FROM cases_llm_groups g WHERE g.case_id = c.id AND g.category = 'cat_migration' AND g.role IN ('participant', 'design_criterion')) THEN c.id END) / COUNT(DISTINCT c.id), 1) as migration
    FROM cases c
    LEFT JOIN cases_methods cm ON c.id = cm.case_id
    JOIN methods m ON m.id = cm.method_id
    GROUP BY m.title
    HAVING COUNT(DISTINCT c.id) >= 20
    ORDER BY total_casos DESC
    LIMIT 12
""", engine)

cats = ['gender', 'race', 'class', 'group_age', 'disability', 'migration']
df_heat = df.set_index('metodo')[cats]
df_heat = df_heat[df_heat.max(axis=1) >= 10]

import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df_heat, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, ax=ax)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('heatmap_metodos.png', dpi=150, bbox_inches='tight')
plt.show()